# U6-L4：动作编排

**课程目标**

1. 能围绕一个展示任务选择动作素材，标明来源、首尾姿态与使用条件，形成3～5步作品脚本。
2. 能把已验证的录制轨迹与内置动作按顺序编排，结合动作实际表现与取消终态判断结果；未确认结束时不重复启动。
3. 能用一次试演记录找出衔接或表达问题，调整一个方案要素并说明复测结果或待验证事项。


## 一、把动作组织成作品

L2保存并回放了自己的轨迹，L3用视频创建了训练任务。本课选择一个具体场景，例如“来宾面前的致意展示”，把已有动作组织成有开始、有内容、有结束的作品。

可以选择录制动作、动作工坊作品和系统内置动作。每个素材是否已经完成、能否在当前设备运行，以及前后姿态是否合适，都要先检查；不能只按动作名称排顺序。

> **怎样让不同来源的动作按预定次序执行，又能在某一步未完成时保留正确的结果？**


## 二、素材来源与执行条件

| 来源 | 本课可使用的材料 | 调用或观察方式 | 编排前检查 |
| --- | --- | --- | --- |
| L2录制轨迹 | 本组已回放验证的`.btraj` | `TrajectoryData.load()`后交给`execute_trajectory()` | K1机型、文件、首尾姿态、单项回放 |
| 内置动作 | `list_actions()`实际列出的动作ID | `do_action(action_id)` | 型号与版本、幅度、时长、可打断性、单项执行 |
| L3动作工坊作品 | 已完成并验证的动作名称与版本 | 使用教师确认可用的部署或调用方式 | 训练/运行验证、机型、资源与首尾姿态 |

本课的Python脚本直接支持前两类素材。动作工坊作品先写入作品设计；程序入口由教师确认并提供后，再编入同一次程序试演。尚未接入的片段标为“待接入”，不可用打印动作名代替执行，也不能拿`.btraj`接口直接执行训练策略资源。

当前脚本不支持的片段若保留在程序清单中，会在开始前报错，不会先执行前面的动作。可以另存一份仅含已接入动作的基础版本，并说明它与完整作品设计的差别。


### 2.1 顺序执行怎样衔接

`execute_trajectory()`和`do_action()`都会返回任务句柄。本课当前实验版本先给动作一段运行时间，再调用`cancel()`请求释放任务；随后有限等待，返回`CANCELLED`才进入下一步。这个状态表示任务已取消，动作是否按设计完成还要结合现场观察。第一次等待达到上限，不会自动停止动作。

确认任务结束只解决了任务衔接条件，没有自动生成两个动作之间的过渡轨迹。上一动作的结束姿态、下一动作的起始姿态，以及动作间是否站稳，仍需单项验证和教师观察。

这些运动任务不同时启动。顺序编排也不等于把多个文件合并为一条轨迹，更不保证任意两个动作都能自然衔接。


## 三、设计与检查作品

先写场景和希望观众看到的内容，再选择3～5步动作。至少包含一段本组录制轨迹和一个教师确认的内置动作；L3作品可列入完整方案，接入条件不足时保留为待验证片段。

| 步骤 | 动作作用 | 来源、文件或ID、版本 | 起始/结束姿态 | 前后衔接检查 | 已验证/待验证 |
| --- | --- | --- | --- | --- | --- |
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

先分别检查素材，再检查相邻动作。需要改变录制动作的首尾姿态时，回到L2重新录制并另存版本，不在未验证的情况下直接改关节数据。


## 四、实践：编排、试演与调整

⚠️ **安全**：作品会连续执行动作，全程由教师看护，人员退出动作范围。使用实体按钮完成准备模式（PREP）到行走模式（WALK）的准备，确认直立稳定后再运行。教师先检查单项动作和相邻衔接；异常时按设备规范中止，不能依赖Python等待超时来停止动作。站立时严禁WALK直接切到DAMP。

📌 **前置**：L2成功回放的文件；U1-L4的内置动作调用；已完成第三节作品设计。L3仍未完成或未接入的作品可保留在设计稿，不冒充本次程序执行成果。


**STEP 01：** 连接设备并查看内置动作

先读取清单。选择教师确认可用的动作，保留清单实际返回的ID，不把整数编号或界面名称自行转换成SDK ID。

`action_ids`用普通循环保存动作ID，供STEP 02核对。`active_motion = None`表示本课还没有发起动作任务；发起任务后，这个变量保存当前句柄。`plan_ready`记录素材是否检查通过，`run_log`保存本轮执行结果。

本格在本课开始时运行一次。中断试演后保留当前连接与句柄，先检查任务和设备，不要靠重跑本格重新初始化。


In [ ]:
from boosteros.robots.booster import BoosterRobot
from boosteros.types import TrajectoryData

robot = BoosterRobot()
active_motion = None
plan_ready = False
run_log = []
actions = robot.list_actions()
action_ids = []
print("机器人：", robot.robot_info.model, robot.robot_info.serial_number)
print("当前模式：", robot.get_mode())
for action in actions:
    action_ids.append(action.id)
    print(f"ID：{action.id:<22}, 动作类型：{action.type}, 动作持续时间：{action.duration}, 动作是否可以中断{action.interruptible}")



### 4.1 写出本次程序执行清单

每个小列表按顺序填写“素材类型、对应文件或ID”。`trajectory`表示录制轨迹，`action`表示内置动作。先按本组方案替换示例；`hand_wave`、`bow`也要在本机清单中存在并经教师确认，才能使用。作品应为3～5步，同时包含本组录制轨迹和内置动作。

**STEP 02：** 检查素材并读入轨迹

本格不执行运动。`step[0]`取素材类型，`step[1]`取文件名或ID。检查通过后，程序把类型、文件名或ID、执行所需数据依次放入`prepared_steps`。内置动作的数据就是ID；录制动作的数据是读回的轨迹对象。

程序每次先将`plan_ready`设为`False`，只有清单内全部素材检查通过，才改为`True`。文件读取报错、ID不存在或素材类型未接入时，先处理问题，不能继续STEP 03。动作工坊片段不能擅自写成`action`或`trajectory`。

修改文件、ID或动作顺序后，重新运行本格，再进行试演。


In [ ]:
plan_ready = False
steps = [
    ["action", "hand_wave"],
    ["trajectory", "my_action_v1.btraj"],
    ["action", "bow"],
]
prepared_steps = []

if len(steps) < 3 or len(steps) > 5:
    print("请将作品安排为3～5步")
else:
    for source, resource in steps:
        if source == "trajectory":
            data = TrajectoryData.load(resource)
            if not data.points or data.duration.seconds <= 0:
                print("轨迹没有有效的动作数据：", resource)
                break
        elif source == "action":
            if resource not in action_ids:
                print("本机动作清单中没有：", resource)
                break
            data = resource
        else:
            print("本次程序未接入这种素材：", source)
            break
        prepared_steps.append([source, resource, data])

    if len(prepared_steps) == len(steps):
        plan_ready = True
        print("全部素材检查通过，尚未发起运动")


### 4.2 试演并记录每一步

`for source, resource, data in prepared_steps`依次取出准备好的动作，并把每项中的素材类型、文件名或ID、执行所需数据分别交给这三个变量。程序根据类型调用`execute_trajectory()`或`do_action()`，把当前任务句柄保存在`active_motion`中。

轨迹先等待其记录时长加1秒，内置动作先等待5秒；教师须按已验证素材核对这些等待参数，不能把它们当作所有动作的完成时间。接着调用`cancel()`请求结束任务，再用`wait(timeout=5.0)`等待取消结果。当前代码取得`CANCELLED`才继续下一步，其他状态均停止后续启动。`CANCELLED`不能证明动作已经完整播放，现场还要记录动作表现。超时不会自动停止机器人，未确认结束时保留句柄，由教师检查。

`try`中的SDK调用若抛出`RuntimeError`，例如机器人正在执行其他动作，`except`会打印错误并用`break`退出循环。文件读取等错误仍保留Python原本的报错信息，便于按提示检查。

`run_log`按执行顺序记录素材来源、文件名或ID、状态。每次试演前先保存上一轮记录；重新运行STEP 03会清空本轮日志。

**STEP 03：** 经教师检查后试演

<font color="red">提示：当前应为行走模式，教师确认清单顺序、动作衔接与周围空间。先完成STEP 02，再运行本格；异常中断后先核对`active_motion`和设备状态，不能直接重跑。</font>


In [ ]:
import time

run_log = []

if not plan_ready:
    print("请先完成STEP 02的素材检查")
elif active_motion is not None and not active_motion.done():
    print("上一任务仍未结束，请教师检查，暂不重新试演")
elif robot.get_mode() != "walk":
    print("请先按教师要求进入行走模式")
else:
    for source, resource, data in prepared_steps:
        try:
            if source == "trajectory":
                active_motion = robot.execute_trajectory(data)
                active_motion.wait(timeout=data.duration.seconds+1.0)
            else:
                active_motion = robot.do_action(data)
                active_motion.wait(timeout=5.0)
            active_motion.cancel()
            status = active_motion.wait(timeout=5.0)
        except RuntimeError as error:
            print("本步发生错误，不启动后续动作：", error)
            run_log.append([source, resource, "调用或等待失败"])
            break

        print(resource, "任务状态：", status)
        run_log.append([source, resource, status])
        if status != "CANCELLED":
            print("本步未取得预期的取消终态，不启动后续动作")
            break


**STEP 04：** 读日志，补充现场观察

每条记录依次保存“来源、素材、结果”。`row[0]`、`row[1]`、`row[2]`取出这三个内容。没有开始调用的步骤不会出现在日志中，应在作品记录里标为“未执行”；“调用或等待失败”也不能记作完成。

即使各步都取得`CANCELLED`，仍需逐项观察动作是否完整、节奏与首尾姿态是否符合设计。任务被取消与动作效果合意应分别记录。


In [ ]:
print("来源 | 素材 | 结果")
for row in run_log:
    print(row[0], row[1], row[2], sep=" | ")


### 4.3 根据试演修改一个要素

选择顺序、素材或某个录制动作的首尾姿态中的一项修改，写出原因。确认上一轮运动任务已结束、教师恢复合适姿态后，重新检查整份素材清单，再试演。

复测使用新版本名并保留原记录。若新增动作工坊片段尚未有可用的程序入口，记录待接入条件；可以在Studio中单独演示该素材，但必须注明“单独运行”，不能说它已经进入这次SDK连续编排。


## 五、顺序、状态与更复杂的组织

本课用列表和循环确定动作顺序，用任务结果决定继续或停止。动作素材可以来自学习算法，运行顺序仍由人编写的规则决定。

U5-L4用“待机、会话已发起、停止请求已发出”描述应用状态。作品若要等待来宾、根据条件选择动作，也可先设计相应状态与转换。行为树则把动作与判断组织为可组合节点，是另一种组织方式。本课不要求实现新的状态机或行为树；作品设计、两类动作混合执行与结果复核是本次实践任务。


## 六、收尾与交付

教师确认运动任务已结束、机器人状态适合收尾后，使用实体按钮退回准备模式（PREP）；待直立稳定，再支撑机器人并按设备规范扶坐、进入阻尼模式（DAMP）。严禁WALK直接切到DAMP。

交付作品脚本、素材清单、试演日志、现场观察及修改说明。演示视频由教师按课堂条件安排，画面注明执行的是基础版本、完整版本还是单独素材，不混用三种成果。


## 七、作品记录与学习评价

### 7.1 试演记录

| 步骤 | 素材与版本 | 任务状态/未执行 | 现场效果 | 需要调整什么 |
| --- | --- | --- | --- | --- |
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

分别填写本组录制动作、内置动作，以及有条件使用的动作工坊作品。未完成训练、未接入程序或未部署时如实标明。

### 7.2 检查点

- 作品是否有明确场景、3～5步设计和可追溯的素材来源。
- 基础程序是否实际包含录制轨迹与内置动作，且按本课版本在确认取消终态后继续，并分别记录动作表现。
- 记录能否区分任务结束方式、动作完整性与衔接效果、未执行步骤。
- 修改建议是否对应试演证据；未复测的改动是否标明待验证。

### 7.3 思考题

1. 第一条轨迹结束姿态与下一动作起始姿态不同，单纯`wait()`能否自动解决？
2. 等待返回`RUNNING`与返回`FAILED`时，下一步都不启动，但两者分别需要处理什么？
3. 动作由强化学习训练得到，整个作品的顺序是否也由模型决定？依据是什么？

### 7.4 学习评价

1. 提交一份至少包含两类可执行素材的作品及试演记录。
2. 指出脚本中保存当前句柄、等待任务和判断是否继续的代码位置，解释各自作用。
3. 说明一个本次修改及其结果；未复测时说明还缺哪项验证。

> 参考答案（教师留存，PDF学生版删除）：1. 以实际文件、ID和记录核对，不将待接入素材当执行成果；2. `active_motion`保存句柄，两次`wait()`分别等待运行与取消结果，`cancel()`提交取消请求，当前版本以`CANCELLED`决定是否继续；`done()`用于重跑前核对，动作完整性另据现场观察；3. 根据观察评价。状态机/行为树不作本课必达技能。
